# Linked Lists


## Topic overview

Sequential pointer chains; most patterns are pointer manipulations.

## Pattern-recognition rules

- Reverse in place.
- Detect cycles (Floyd's).
- Merge sorted lists.
- Dummy-head to simplify insertion.

## Common data structures

- Singly linked
- Doubly linked

## Standard complexity expectations

- Almost always O(n) time, O(1) extra space.

## Common mistakes

- Losing the head pointer.
- Not handling the empty list.

## Original illustrative example

In [ ]:
# Replace with an ORIGINAL example. Do not paste external
# problem statements. See src/algorithms/ for reusable helpers.
example_input = []
example_expected = None

## Add solved problems below

Each new sub-section should follow the template in `../templates/notebook_template.ipynb`.

1. [Reverse Linked List](#reverse-linked-list)


# Reverse Linked List

## Metadata

- Source: NeetCode / LeetCode 206
- Problem URL: https://leetcode.com/problems/reverse-linked-list/
- Difficulty: Easy
- Topic: Linked Lists
- Language: C++ (Python below is the same pointer walk, so this notebook can run it)
- Date started: 2026-10-04
- Date solved: 2026-10-04
- Current mastery level: 1
- Last reviewed: 2026-10-04
- Next review:


## Problem statement in my own words

You are given the head of a singly linked list. Reverse the links and
return the new head.

```
head
 ↓
[0] → [1] → [2] → [3] → nullptr

becomes

                new head
                   ↓
nullptr ← [0] ← [1] ← [2] ← [3]
```

The nodes stay where they are. Only each `next` pointer changes direction.
An empty list stays empty.


## Inputs, outputs, and constraints

- Input: `ListNode* head`
- Output: `ListNode*` — the new beginning
- Constraints:
  - $0 \le n \le 1000$
  - $-1000 \le$ `Node.val` $\le 1000$

NeetCode's node is a pointer type:

```cpp
struct ListNode {
    int val;
    ListNode* next;

    ListNode(int val) {
        this->val = val;
        next = nullptr;
    }
};
```

In C++, a pointer stores an address. `next` is not the next value. It is
the address of the next node, or `nullptr` when the chain ends.


## Examples

| Input | Expected | Notes |
|---|---|---|
| `head = [0, 1, 2, 3]` | `[3, 2, 1, 0]` | every link flips |
| `head = []` | `[]` | `head` is already `nullptr` |
| `head = [7]` | `[7]` | one node, `next` was already null |


## The danger

Suppose the walk has already reversed `0`, and `curr` is `1`:

```
prev       curr
 ↓           ↓
[0]         [1] → [2] → [3] → nullptr
```

If you write `curr->next = prev` immediately, node `1` now points at
`0`, and the address of `2` is gone. The rest of the list is leaked.

Save the forward link first.

```cpp
ListNode* prev = nullptr;  // reversed portion
ListNode* curr = head;     // node being processed
ListNode* next = nullptr;  // saved successor
```

| Pointer | Job |
|---|---|
| `prev` | head of the already-reversed prefix |
| `curr` | node we are about to flip |
| `next` | where `curr` used to point, so the walk can continue |


## The four moves

At every node, in this order:

1. Save `curr->next`
2. Reverse `curr->next` so it points at `prev`
3. Move `prev` onto `curr`
4. Move `curr` onto the saved node

```text
prev = nullptr
curr = head

while curr exists:
    next = curr.next
    curr.next = prev
    prev = curr
    curr = next

return prev
```

Return `prev`, not `curr`. When the loop stops, `curr` is `nullptr` and
`prev` is the old tail, which is the new head.

## Trace — `[0, 1, 2, 3]`

Start: `prev = nullptr`, `curr = 0`.

| Step | save | after reverse | then |
|---|---|---|---|
| 1 | `next = 1` | `nullptr ← [0]` | `prev = 0`, `curr = 1` |
| 2 | `next = 2` | `[0] ← [1]` | `prev = 1`, `curr = 2` |
| 3 | `next = 3` | `[1] → [0] → nullptr`, `prev = 2` | `curr = 3` |
| 4 | `next = nullptr` | `[3] → [2] → [1] → [0] → nullptr` | `curr = nullptr` |

The loop invariant, checked before each iteration:

> `prev` is a correctly reversed list of every node already processed.
> `curr` is the first node not yet processed.

Midway:

```
REVERSED                 NOT YET
[1] → [0] → nullptr      [2] → [3] → nullptr
 ↑                        ↑
prev                     curr
```

One iteration pulls `2` off the unprocessed side and puts it at the
front of the reversed side. When `curr` becomes `nullptr`, `prev` is
the whole list.


## C++ solution

```cpp
class Solution {
public:
    ListNode* reverseList(ListNode* head) {
        // Already-reversed portion. Empty at the start.
        ListNode* prev = nullptr;

        // Node we are about to flip.
        ListNode* curr = head;

        while (curr != nullptr) {
            // 1. Save the forward link before it is overwritten.
            ListNode* next = curr->next;

            // 2. Point this node at the reversed prefix.
            curr->next = prev;

            // 3. The reversed prefix now starts here.
            prev = curr;

            // 4. Continue from the saved successor.
            curr = next;
        }

        // prev is the old tail, which is the new head.
        return prev;
    }
};
```

Do not memorize the function. Memorize the four lines:

```cpp
ListNode* next = curr->next;  // save
curr->next = prev;            // reverse
prev = curr;                  // advance prev
curr = next;                  // advance curr
```

Save, reverse, advance, advance. The same four moves show up inside
Reverse Linked List II, Reverse Nodes in k-Group, and Reorder List.


## Complexity

Let $n$ be the number of nodes. Each node is visited once.

$$
T(n) = O(n)
$$

Only three pointers are stored, however long the list is. That beats
the recursive reversal, which uses $O(n)$ call-stack space.

$$
S(n) = O(1)
$$

## Edge cases

- Empty list: `curr` starts as `nullptr`, the loop never runs, and
  `prev` is returned as `nullptr`. No extra `if`.
- One node: save `nullptr`, point `next` at `nullptr`, return that node.
- Two nodes: one real swap of the single link.
- The values never matter. Only addresses move.


## Testing

The checks use the same four assignments as the C++ loop. This kernel
is Python, so the mirror is what actually runs.


In [ ]:
class ListNode:
    def __init__(self, val: int = 0, next: "ListNode | None" = None) -> None:
        self.val = val
        self.next = next


def from_list(values: list[int]) -> ListNode | None:
    dummy = ListNode(0)
    tail = dummy
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next


def to_list(head: ListNode | None) -> list[int]:
    out: list[int] = []
    while head is not None:
        out.append(head.val)
        head = head.next
    return out


def reverse_list(head: ListNode | None) -> ListNode | None:
    prev = None
    curr = head
    while curr is not None:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
    return prev


assert to_list(reverse_list(from_list([0, 1, 2, 3]))) == [3, 2, 1, 0]
assert to_list(reverse_list(from_list([]))) == []
assert to_list(reverse_list(from_list([7]))) == [7]
assert to_list(reverse_list(from_list([1, 2]))) == [2, 1]

print("Reverse Linked List checks passed")


## Visualization

Nodes stay in their original places. Only the arrows flip. That is the
point of the algorithm: we are not moving values into a new list.

- orange: `curr`, the node just reversed in this frame
- gold: the saved `next`
- navy: `prev`
- green: the rest of the already-reversed chain
- gray: `nullptr`

If `ipywidgets` is installed, use **Next step** / **Prev** / **Reset**.
Otherwise every iteration is drawn as its own frame.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch


@dataclass(frozen=True)
class ReverseFrame:
    step: int
    values: List[int]
    nxt: Dict[int, Optional[int]]
    prev: Optional[int]
    curr: Optional[int]
    saved: Optional[int]
    decision: str


def reverse_frames(values: List[int]) -> List[ReverseFrame]:
    """Record each save / reverse / advance, matching the C++ loop."""
    n = len(values)
    nxt: Dict[int, Optional[int]] = {i: (i + 1 if i + 1 < n else None) for i in range(n)}
    prev: Optional[int] = None
    curr: Optional[int] = 0 if n else None
    frames: List[ReverseFrame] = [
        ReverseFrame(0, list(values), dict(nxt), prev, curr, None, "start: prev = nullptr, curr = head")
    ]
    if n == 0:
        return frames
    step = 1
    while curr is not None:
        saved = nxt[curr]
        nxt[curr] = prev
        frames.append(
            ReverseFrame(
                step,
                list(values),
                dict(nxt),
                prev,
                curr,
                saved,
                f"save next, reverse node {values[curr]}, then advance",
            )
        )
        prev = curr
        curr = saved
        step += 1
    frames.append(
        ReverseFrame(
            step,
            list(values),
            dict(nxt),
            prev,
            curr,
            None,
            "curr is nullptr — return prev (the new head)",
        )
    )
    return frames


def _node_color(idx: int, frame: ReverseFrame) -> str:
    if frame.curr == idx:
        return "#e76f51"
    if frame.saved == idx and frame.curr is not None:
        return "#e9c46a"
    if frame.prev == idx:
        return "#1d3557"
    # Nodes already reversed sit on the chain that starts at prev.
    seen = set()
    walk = frame.prev
    while walk is not None and walk not in seen:
        seen.add(walk)
        walk = frame.nxt[walk]
    if idx in seen:
        return "#2a9d8f"
    return "#4c78a8"


def plot_reverse_frame(frame: ReverseFrame, *, ax=None):
    """Nodes stay put. Only the next arrows change direction."""
    created = ax is None
    values = frame.values
    n = len(values)
    if created:
        _, ax = plt.subplots(figsize=(max(8, (n + 2) * 1.7), 3.6))

    ax.set_xlim(-2.1, max(n, 1) * 1.7 + 1.6)
    ax.set_ylim(-1.5, 2.2)
    ax.axis("off")
    ax.set_title(f"Step {frame.step}   {frame.decision}", loc="left", fontsize=11)

    if n == 0:
        ax.text(0, 0.4, "head = nullptr    return nullptr", fontsize=13, color="#1d3557")
        return ax

    xs = [i * 1.7 for i in range(n)]

    def box(x: float, y: float, label: str, face: str, text_color: str = "white") -> None:
        ax.add_patch(
            FancyBboxPatch(
                (x - 0.42, y - 0.32),
                0.84,
                0.64,
                boxstyle="round,pad=0.02,rounding_size=0.08",
                facecolor=face,
                edgecolor="#1d3557",
                linewidth=1.2,
            )
        )
        ax.text(x, y, label, ha="center", va="center", fontsize=12, fontweight="bold", color=text_color)

    box(-1.3, 0.4, "null", "#6c757d")
    for i, value in enumerate(values):
        face = _node_color(i, frame)
        text_color = "#1d3557" if face == "#e9c46a" else "white"
        box(xs[i], 0.4, str(value), face, text_color)
        ax.text(xs[i], -0.15, f"[{i}]", ha="center", va="top", fontsize=8, color="#6b6b6b")

    for i in range(n):
        target = frame.nxt[i]
        x0 = xs[i]
        if target is None:
            # Node 0's null is the left sentinel. Any other null is the
            # still-unreversed tail, drawn to the right of the last node.
            if i == 0:
                start = (x0 - 0.46, 0.4)
                end = (-1.3 + 0.48, 0.4)
            else:
                x1 = xs[-1] + 1.35
                box(x1, 0.4, "null", "#6c757d")
                start = (x0 + 0.46, 0.4)
                end = (x1 - 0.48, 0.4)
            ax.add_patch(
                FancyArrowPatch(
                    start,
                    end,
                    arrowstyle="-|>",
                    mutation_scale=12,
                    color="#1d3557",
                    linewidth=1.3,
                    connectionstyle="arc3,rad=0.0",
                )
            )
        else:
            x1 = xs[target]
            backward = target < i
            rad = 0.28 if backward else -0.22
            y = 0.78 if backward else 0.4
            ax.add_patch(
                FancyArrowPatch(
                    (x0 + (-0.46 if backward else 0.46), y),
                    (x1 + (0.46 if backward else -0.46), y),
                    arrowstyle="-|>",
                    mutation_scale=12,
                    color="#1b6b62" if backward else "#1d3557",
                    linewidth=1.4,
                    connectionstyle=f"arc3,rad={rad}",
                )
            )

    def tag(idx: Optional[int], name: str, y: float, color: str) -> None:
        if idx is None:
            if name == "prev":
                ax.text(-1.3, 1.15, name, ha="center", va="bottom", fontsize=10, fontweight="bold", color=color)
            else:
                ax.text(
                    xs[-1] + 1.15,
                    1.55,
                    f"{name} = nullptr",
                    ha="left",
                    fontsize=10,
                    fontweight="bold",
                    color=color,
                )
            return
        ax.annotate(
            name,
            xy=(xs[idx], 0.76),
            xytext=(xs[idx], y),
            ha="center",
            fontsize=10,
            fontweight="bold",
            color=color,
            arrowprops={"arrowstyle": "->", "color": color, "lw": 1.0},
        )

    tag(frame.prev, "prev", 1.7, "#1d3557")
    tag(frame.curr, "curr", 1.35, "#9b2226")
    if frame.saved is not None and frame.curr is not None:
        tag(frame.saved, "next", 1.95, "#b08900")

    if created:
        plt.tight_layout()
    return ax


def visualize_reverse_all(values: List[int]) -> None:
    frames = reverse_frames(values)
    print(f"values={values}")
    for frame in frames:
        prev = "null" if frame.prev is None else frame.values[frame.prev]
        curr = "null" if frame.curr is None else frame.values[frame.curr]
        print(f"step {frame.step}: prev={prev} curr={curr}  {frame.decision}")
        plot_reverse_frame(frame)
    plt.show()


def step_through_reverse(values: List[int]) -> None:
    frames = reverse_frames(values)
    try:
        import ipywidgets as widgets
        from IPython.display import clear_output, display
    except ImportError:
        visualize_reverse_all(values)
        return

    output = widgets.Output()
    state = {"i": 0}

    def render() -> None:
        with output:
            clear_output(wait=True)
            frame = frames[state["i"]]
            print(f"frame {state['i'] + 1}/{len(frames)}")
            plot_reverse_frame(frame)
            plt.show()

    def on_next(_=None) -> None:
        state["i"] = min(state["i"] + 1, len(frames) - 1)
        render()

    def on_prev(_=None) -> None:
        state["i"] = max(state["i"] - 1, 0)
        render()

    def on_reset(_=None) -> None:
        state["i"] = 0
        render()

    prev_btn = widgets.Button(description="Prev")
    next_btn = widgets.Button(description="Next step")
    reset_btn = widgets.Button(description="Reset")
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    reset_btn.on_click(on_reset)
    display(widgets.HBox([prev_btn, next_btn, reset_btn]), output)
    render()

print("Example — reverse [0, 1, 2, 3]")
step_through_reverse([0, 1, 2, 3])

print("\nEmpty list")
step_through_reverse([])


## Alternative approaches

- Recursion: reverse the tail, then point `head->next->next = head`
  and `head->next = nullptr`. Same $O(n)$ time, $O(n)$ stack.
- Copy values into a vector, reverse the vector, write them back.
  $O(n)$ extra memory, and it hides the pointer lesson.
- Stack of node pointers: also $O(n)$ extra memory.

## Mistakes I made

- Reversing `curr->next` before saving it, which drops the rest of the list.
- Returning `curr` or `head`. Both are wrong at the end: `curr` is null,
  and the original `head` is now the tail.
- Special-casing the empty list. The loop already handles it.
- Writing `next` as a name that shadows something, then using the field
  `curr->next` after it was overwritten. The local copy has to be a
  different pointer.

## Pattern recognition

Singly linked reversal is always:

```cpp
ListNode* next = curr->next;
curr->next = prev;
prev = curr;
curr = next;
```

If a later problem reverses only part of a list, the same four lines
run between two fixed endpoints. The endpoints are what change, not
the flip.

## Related problems

- Reverse Linked List II — reverse one index range, then stitch it back
- Reverse Nodes in k-Group — this loop once per block of $k$
- Swap Nodes in Pairs — reversal with $k = 2$
- Reorder List — reverse the second half, then merge
- Palindrome Linked List — reverse half, then compare

## Real-world or engineering connection

Any singly linked buffer (free lists, intrusive lists, packet chains)
is reversed by retargeting `next`, not by sliding the payloads. Losing
the successor pointer is a use-after-free waiting to happen: you still
need that address to walk forward.

## Final takeaways

1. A `next` pointer is an address, not a value.
2. Save that address before you overwrite it.
3. Four moves: save, reverse, advance `prev`, advance `curr`.
4. Return `prev`.


## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| 2026-10-04 | 1 | First write-up: iterative C++ reversal, save before reverse |
